# Claude Developer Platform：ビルドアロング

**レベル：** 300（中級） | **モデル：** Claude Sonnet 5（`claude-sonnet-5`）

## 作るもの

チケットの詳細を読み、ナレッジベースを検索し、構造化された解決結果を出力する、マルチツールの**サポートチケットエージェント**です。フレームワークは使わず、Claude APIを直接呼び出します。

**お客様のシナリオ：** TechFlow（中堅B2B SaaS、1日に500件以上のチケット）は、Tier 1の一次対応を自動化し、人間の担当者が複雑なエスカレーションに集中できるようにしたいと考えています。

## 学習目標

1. マルチツールのエージェントループ（`while stop_reason == "tool_use"`）を実装する
2. 構造化出力とツール使用を、1つのエージェントの中で組み合わせる
3. アダプティブ思考と`output_config.effort`を組み合わせ、推論の深さを制御する
4. 思考、ツール呼び出し、応答をリアルタイムにストリーミングする

## どこに位置づけられるか

Claudeで開発する方法はいくつかあります。このセッションでは**Messages API**を扱います。最も低レベルで、リクエストとレスポンスのやり取りを完全に制御できます。

| サーフェス | 概要 | 使う場面 |
|---------|-----------|----------------|
| **Messages API** ← *このセッション* | Claudeへの直接のHTTP／SDK呼び出し | エージェントループの完全な制御、独自のオーケストレーション、本番バックエンド |
| **Agent SDK** | ツールのディスパッチを組み込んだPythonフレームワーク | エージェントの素早いプロトタイプ、オーケストレーションを任せたいとき |
| **Claude Code** | CLIベースのコーディングエージェント | 開発者の生産性向上、リポジトリ単位のタスク、対話的なコーディング |
| **claude.ai / Claude for Enterprise** | プロジェクトやMCPを備えたチャットインターフェース | エンドユーザーのワークフロー、企業のナレッジワーク、開発者以外の用途 |

今日は、素のループを自分で作ります。ここを理解しておくと、上位の層がすべて分かりやすくなります。

---

**Partner Basecamp参加者向けに作成。** 研修資料としての複製・再配布は禁止です。ここで学ぶパターンを、お客様のご支援に活用することは自由です。

## セットアップ

以下のセルを実行して、依存パッケージのインストール、APIキーの設定、モックデータの読み込みを行います。

- **APIキー**は、セットアップのセルが作成する`.env`ファイルに貼り付けます。一度貼れば十分です（gitignore対象で、カーネルを再起動しても残ります）。
- **✏️ YOUR TURN（あなたの番）のセル**が、今日コードを書く場所です。
- **APIセルは時間がかかります**。思考ありのエージェント実行には30〜90秒かかることがあります。セルの横の`[*]`は、まだ処理中であることを示します。


### セットアップ：Claudeに接続する

最初に次のセルを実行します。セットアップのセルは、初回実行時に**`.env`ファイル**を作成します（gitignore対象なので、キーがコミットされることはありません）。ファイルを開き、`ANTHROPIC_API_KEY=`の後ろにキーを貼り付けて保存し、もう一度実行してください。カーネルを再起動しても残るので、貼るのは一度だけです。*（まだ`.env`がない場合は、代わりに非表示の入力ボックスが表示されます。）* 緑色の**「✓ API key verified」**バナーが出れば完了です。赤いバナーが出たら、表示された指示に従って、もう一度セルを実行してください。

In [ ]:
# ── Install と Import ──
# 依存パッケージをこのカーネルにインストールします。再実行しても安全で、PEP 668で制限されたPythonでも動作します。
import importlib.util, os, subprocess, sys

# ── Environment guard（環境ガード） ──────────────────────────────
# 下のinstallerと同じセルに置いているので、スキップできません。
# システムのPythonにパッケージが直接インストールされることは決してありません
# （以前の最悪のケースは、IT部門が管理するマシンに対する--break-system-packagesでした）。ここで止まった場合は、
# SETUP.mdの「Why the notebook just stopped」を参照してください。
def _in_isolated_env():
    """実行中のカーネルが、venv／virtualenv／conda環境、またはColabであればTrueを返します。
    判定にはインタープリタ自身（sys.*）を使います。起動元のシェルから引き継いだ
    有効化用の環境変数は、sys.executableが実際にその環境の内部にある場合にだけ
    信頼します。有効化済みのターミナルから起動したシステムPythonのカーネルも
    VIRTUAL_ENVを引き継ぎますが、それを通してはいけません。"""
    if "google.colab" in sys.modules:
        return True  # Colabは使い捨ての独自ランタイムでサンドボックス化されている
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405のvenv（python -m venv）。ほとんどのconda環境も該当
    if hasattr(sys, "real_prefix"):
        return True  # 従来のvirtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # shellからの伝聞にすぎない。kernelは別の場所にある
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # condaの共有`base`は、分離された環境として扱わない
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  環境ガードをバイパスしました（BASECAMP_ALLOW_SYSTEM_PYTHON=1）。"
          "意図的にこのPythonへインストールします。")
elif not _in_isolated_env():
    _HEAD = "✗ システムのPythonを検出：何もインストールせずに停止しました"
    _BODY = (
        "ここにパッケージをインストールすると、マシン全体のPythonが変わります。\n"
        "会社管理のノートPCでは、ITへの問い合わせが必要になることもあります。\n"
        "\n"
        "対処（2ステップ）：\n"
        "  1. ターミナルで、リポジトリのルートから次を実行します：\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windowsの場合： .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. VS Codeで、カーネル名（右上）をクリック → Select Another Kernel →\n"
        "     Python Environments → 末尾が .venv のものを選び → このセルをもう一度実行します。\n"
        "\n"
        "condaを使っている場合は、`conda activate <env>`（`base`は不可）を実行してから、そのカーネルを選びます。\n"
        "自己管理のマシンを使うファシリテーターは、BASECAMP_ALLOW_SYSTEM_PYTHON=1 で回避できます。"
    )
    _shown = False
    try:  # 下のAPIキー確認と同じバナー表示（緑のボックス、赤のバリエーション）
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "環境ガードがこのセルを停止しました。上のメッセージを確認してください。"
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ 環境ガード：分離されたインタープリタを検出しました。インストールしても安全です")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements：(import_name, pip_spec)のリスト。不足しているものだけを、
    実行中のインタープリタにインストールします。まず通常のインストール、次にユーザー領域、
    最後にPEP 668の上書き（ユーザー領域が先、システム全体は最後の手段）の順に試します。
    どの試行も出力は抑えられ、pipの出力はストリーム表示されずに取り込まれるため、
    ロックダウンされたPython（HomebrewやDebian、PEP 668）でも、フォールバックで
    成功した場合に「externally-managed-environment」の長いエラーが出ることはありません。
    すべての方法が失敗したときだけ、生のトレースバックではなく、
    venvでの対処法とともに原因を表示します。"""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + " — first run only, please wait…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "(pipからの出力なし)"
    raise SystemExit(
        "\n  インストールできませんでした：" + ", ".join(missing) + "\n"
        "  このPythonはロックダウンされている（PEP 668）か、オフラインです。最も簡単な対処はvenvです：\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windowsの場合： SETUP.md を参照\n"
        f"      pip install {' '.join(missing)}\n"
        "  その後、VS Code（右上のカーネルピッカー）で.venvのインタープリタを選び、Run Allを実行します。\n"
        "  社内プロキシやPyPIのブロックが原因の場合は、リポジトリ直下のSETUP.mdを参照してください。\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ 依存パッケージの準備が完了しました")

import anthropic
import json
import time
import os

# ── APIキーの設定 ──
import os

def _status(ok, msg):
    """ノートブックでは緑／赤のバナー、スクリプトとして実行した場合はプレーンテキストで表示します。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("not in a notebook kernel - use the plain-text banner")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Claudeへの接続：Anthropic APIまたはAmazon Bedrock ──
# どちらのcredentialでも動作します。セルが自動でどちらを持っているかを判別します。
#   Anthropic API：ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock：AWS_BEARER_TOKEN_BEDROCK=...  と  AWS_REGION=us-east-1
# 使うほうを、このセルが作成する.envファイルに入れてください（gitignore対象で、コミットされません）。
# シェルでexportしても構いません。シェルの値は.envファイルより優先されます。
_ENV_TEMPLATE = (
    "# Anthropic APIキー：=の後ろに貼り付けます（引用符やスペースは不要）。保存してから、\n"
    "# セットアップのセルをもう一度実行してください。キーは https://console.anthropic.com/ で取得できます\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- 代わりにAmazon Bedrockを使う場合は、上の行をコメントアウトして、次を入力します：\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """作業ディレクトリから上に向かって、最も近い既存の.envを探します（ルートの.env1つで
    すべての演習をまかなえるようにするため）。まだ存在しなければ、リポジトリのルートを指します。
    ノートブックを単体で開いている場合は、このフォルダーです。"""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# 小さな.envパーサー（python-dotenvへの依存なし）。実行のたびに読み直すので、キーを貼って
# 再実行すれば反映されます。環境（シェル／Claude Code／CI）にある実際の値が
# 優先され、placeholderが残ることはありません。
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """警告の黄色いバナーを出して停止します。数セル先ではなく、ここでセットアップを失敗させるためです。"""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("認証情報が見つかりません。上のメッセージを確認してください。")


if PROVIDER is None:
    _needs_credentials(
        "📋 続けるには認証情報を追加してください",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Anthropic APIを使う場合は、次を設定します：\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Amazon Bedrockを使う場合は、次の両方を設定します：\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # モデルを有効にしているリージョン\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrockにはリージョンが必要です",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "Bedrockのモデルを有効にしているリージョンを追加してください。例：\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )


def _model(name):
    """BedrockのモデルIDには`anthropic.`というプレフィックスが付きます。Anthropic APIでは、プレフィックスなしのIDを使います。"""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# この演習で使うモデルに名前を付けます。使っているproviderに合わせて解決されます。
MODEL = _model("claude-sonnet-5")        # この演習の主力model
FAST_MODEL = _model("claude-haiku-4-5")  # 低コストで高速（connection checkと採点用）
BIG_MODEL = _model("claude-opus-4-8")    # より大きなmodelを試したいとき


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# Connection check：credentialと、このmodelに実際にアクセスできることの両方を確認します。
# Bedrockでは、キーが有効でも、お使いのアカウント／リージョンでモデルが有効になっていなければ404になります。
# そのため、認証情報の形式だけでなく、実際のモデルIDに対してpingを送ります。
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("モデルを利用できません。上のメッセージを確認してください。")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "Bedrockのキーが拒否されました。AWS_BEARER_TOKEN_BEDROCKと、"
                       "Bedrockの呼び出し権限が付いているかを確認してから、このセルをもう一度実行してください。")
    else:
        _status(False, "キーが拒否されました。このセルをもう一度実行し、キー全体を貼り付けてください"
                       "（sk-ant-で始まります）。")
    raise SystemExit("認証情報が受け付けられませんでした。このセルをもう一度実行して、やり直してください。")
except Exception as exc:
    _status(False, "APIに接続できませんでした（" + type(exc).__name__ + "）。"
                   "ネットワーク接続を確認してから、このセルをもう一度実行してください。")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # 以降のセルや!pythonがこの値を使う
        _status(True, "API key verified（APIキーを確認しました）- Claudeに接続されました。")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# 実際に使うクライアント。タイムアウトを長めにしているのは、ストリーミングなしの呼び出しでmax_tokensが21333を超える場合に必要なためです。
client = _make_client(timeout=900.0)


In [ ]:
# ── Sample ticket データ ──

TICKETS = {
    "TKT-1042": {
        "id": "TKT-1042", "customer": "Acme Corp", "priority": "high",
        "product_area": "billing",
        "description": "3月分の請求書で、二重に請求されました。請求書 #INV-2024-0342 は$4,500ですが、銀行の明細には3月3日付で同額の請求が重複して記録されています。重複分を至急返金してください。",
        "status": "open"
    },
    "TKT-1043": {
        "id": "TKT-1043", "customer": "DataFlow Inc", "priority": "medium",
        "product_area": "api",
        "description": "昨日APIキーをローテーションしてから、Webhookのエンドポイントがイベントを受信しなくなりました。新しいキーがREST呼び出しでは使えることは確認済みですが、Webhookは依然として失敗します。Webhookのログに401エラーが出ています。",
        "status": "open"
    },
    "TKT-1044": {
        "id": "TKT-1044", "customer": "CloudScale Ltd", "priority": "low",
        "product_area": "feature_request",
        "description": "ダッシュボードに一括エクスポート機能がほしいです。現在はレポートを1件ずつエクスポートする必要があり、50件以上のプロジェクトにまたがる四半期サマリーが必要なときは大変な手間です。",
        "status": "open"
    },
    "TKT-1045": {
        "id": "TKT-1045", "customer": "SecureNet Systems", "priority": "critical",
        "product_area": "account",
        "description": "MFAの認証に失敗し続けて、管理者アカウント（admin@securenet.io）がロックアウトされました。SSOがこの管理者アカウントに紐づいているため、47人のチームメンバーがプラットフォームにアクセスできません。すべての業務が止まっています。",
        "status": "open"
    },
    "TKT-1046": {
        "id": "TKT-1046", "customer": "MedTech Solutions", "priority": "high",
        "product_area": "api",
        "description": "昨夜の午前2時ごろから、本番環境の連携で断続的に500エラーが返るようになりました。APIコールの約15%が失敗しています。当社側では何も変更していません。エラーはランダムに見え、同じリクエストでも再試行すると成功することがあります。シンガポールのチームが作業できず、至急解決が必要です。",
        "status": "open"
    },
}

KB_ARTICLES = {
    "KB-001": {"title": "Processing Duplicate Payment Refunds", "content": "For duplicate charges: 1) Verify the duplicate in the billing system, 2) Issue refund through the payment processor (takes 3-5 business days), 3) Send confirmation email with refund reference number. Escalate if amount exceeds $10,000."},
    "KB-002": {"title": "Webhook Authentication After Key Rotation", "content": "When API keys are rotated, webhook signing secrets must also be updated. Go to Settings > Webhooks > Edit endpoint, and regenerate the signing secret. The old secret is invalidated immediately on key rotation. Common mistake: rotating the API key but not the webhook signing secret."},
    "KB-003": {"title": "Bulk Export Feature (Roadmap)", "content": "Bulk export is on the Q3 roadmap. Workaround: Use the REST API's /reports/export endpoint with date range parameters to programmatically export multiple reports. See API docs for batch export examples."},
    "KB-004": {"title": "Admin Account Lockout Recovery", "content": "For locked admin accounts: 1) Verify identity through the secondary email on file, 2) Reset MFA through the admin recovery flow at /admin/recover, 3) Temporary access can be granted through support-level override (requires manager approval). Critical: If SSO is blocked, enable the bypass login at /login/direct for affected users."},
    "KB-005": {"title": "API Rate Limiting Best Practices", "content": "Default rate limits: 100 requests/minute for standard plans, 1000/minute for enterprise. Use exponential backoff with jitter for retries. Monitor usage via the X-RateLimit headers in responses."},
    "KB-006": {"title": "Invoice Discrepancy Resolution", "content": "For billing discrepancies: Check the billing audit log for the account, compare with payment processor records, and verify no pending transactions. Contact finance team for adjustments over $5,000."},
    "KB-007": {"title": "Intermittent 500 Errors Troubleshooting", "content": "For intermittent server errors: 1) Check the status page for known outages, 2) Review rate limit headers - 429s can masquerade as 500s behind load balancers, 3) Check if errors correlate with payload size or specific endpoints, 4) Enable request ID logging and contact support with specific request IDs for investigation. If >10% error rate persists for >1 hour, escalate to engineering."},
}

def get_ticket(ticket_id: str) -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        return json.dumps(ticket)
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

def search_kb(query: str) -> str:
    query_lower = query.lower()
    results = []
    for article_id, article in KB_ARTICLES.items():
        if any(word in article["title"].lower() or word in article["content"].lower()
               for word in query_lower.split() if len(word) > 2):
            results.append({"id": article_id, **article})
    if not results:
        results = [{"id": "KB-000", "title": "No matches found", "content": "No relevant articles found. Consider escalating to Tier 2 support."}]
    return json.dumps(results[:3])

def resolve_ticket(ticket_id: str, resolution: str, status: str = "resolved") -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        ticket["status"] = status
        ticket["resolution"] = resolution
        return json.dumps({"success": True, "ticket_id": ticket_id, "new_status": status})
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

TOOL_FUNCTIONS = {"get_ticket": get_ticket, "search_kb": search_kb, "resolve_ticket": resolve_ticket}

def execute_tool(name: str, input_data: dict) -> str:
    func = TOOL_FUNCTIONS.get(name)
    if func:
        return func(**input_data)
    return json.dumps({"error": f"Unknown tool: {name}"})

print("モックツールとサンプルデータを読み込みました！")
print(f"   Available tickets: {', '.join(TICKETS.keys())}")
print(f"   Knowledge base articles: {len(KB_ARTICLES)}")

---
# パート1：マルチツールのエージェントループ

TechFlowのTier 1サポートチームは、現在すべてのチケットを手作業で処理しています。顧客を調べ、ナレッジベースを検索し、問題を分類し、解決内容の下書きを書きます。1日に500件以上、1件あたり約8分かかります。これをClaudeに自律的に任せたいと考えています。

その作業を置き換えるエージェントを作ります。ツール3つ、ループ1つ、フレームワークなしです。

> **重要な概念：** Claude Sonnet 5は*アダプティブ思考*をサポートしており、タスクの複雑さに応じて、どれだけ推論するかを自動で決めます。すべてのAPI呼び出しで`thinking={"type": "adaptive"}`を指定して有効にします。

## パート2：ツールスキーマを定義する

TechFlowのサポートエージェントには、3つのシステムへのアクセスが必要です。チケット管理プラットフォーム（詳細の参照）、ナレッジベース（解決策の検索）、解決エンジン（チケットのクローズ）です。それぞれがツールスキーマになり、何が使えるか、どう呼び出すかをClaudeに伝えます。

`description`フィールドは重要です。Claudeがあるステップで*どの*ツールを使うかを決める手がかりになるからです。説明があいまいだと、間違ったツールが選ばれます。

> 📖 **参考：** [ツール使用のドキュメント](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview)

### ✏️ YOUR TURN（あなたの番）：`search_kb`と`resolve_ticket`のツールスキーマを完成させる

`get_ticket`は、参考として記入済みです。`<-- fill this in`と書かれた箇所で、空の`description`文字列と`status`のenumを埋めてください。

In [ ]:
# TODO: search_kbとresolve_ticketのツールスキーマを定義する
# 各ツールに必要なもの：name、description、input_schema（propertiesとrequired付き）

tools = [
    # ✅ 例：get_ticketは完成済み。次の2つの参考にする
    {
        "name": "get_ticket",
        "description": "サポートチケットの詳細（顧客、優先度、製品領域、説明）を、IDを指定してすべて取得する。",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string", "description": "チケットID（例：TKT-1042）"}
            },
            "required": ["ticket_id"]
        }
    },

    # ✏️ YOUR TURN（あなたの番）：search_kbを埋める
    # ヒント：descriptionは、Claudeがこのツールを「いつ」呼ぶかを決める手がかりになる。具体的に書く
    {
        "name": "search_kb",
        "description": "",  # <-- fill this in（ここを埋める）
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": ""}  # <-- descriptionを埋める
            },
            "required": ["query"]
        }
    },

    # ✏️ YOUR TURN（あなたの番）：resolve_ticketを埋める
    # ヒント：statusはenumにする。解決状態として考えられる3つは何か？
    {
        "name": "resolve_ticket",
        "description": "",  # <-- fill this in（ここを埋める）
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string"},
                "resolution": {"type": "string"},
                "status": {"type": "string", "enum": []}  # <-- enumの値を埋める
            },
            "required": ["ticket_id", "resolution", "status"]
        }
    }
]

print(f"Defined {len(tools)} tool schemas: {[t['name'] for t in tools]}")

## パート3：エージェントループを作る

ここが自動化の中核です。チケットが届いたら、Claudeがチケットを調べ、解決策を検索し、解決する。この流れを人の手を介さずに実行します。それを可能にするのがエージェントループです。`while response.stop_reason == "tool_use"`の間、ツール呼び出しを取り出し、実行し、結果を追加して、もう一度APIを呼び出します。順序を決めるのはClaudeで、コードはそれを取りまとめるだけです。

**ポイント：** `response.content`は、そのままの形で渡し直します。tool_useブロックと一緒に、thinkingブロックが含まれている場合があります。Claude Sonnet 5では、すべての呼び出しで`thinking={"type": "adaptive"}`を使います。

> 📖 **参考：** [エージェント型ツール使用のパターン](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/agentic-tool-use)

### ✏️ YOUR TURN（あなたの番）：エージェントループを作る

下の`run_agent()`にある`___`の空欄をすべて埋めてください。ループ条件、ツール結果の受け渡し、続きのAPI呼び出しの3か所です。

In [ ]:
SYSTEM_PROMPT = """あなたは、TechFlowのTier 1サポートエージェントです。TechFlowは、中堅企業にプロジェクト管理とチーム協業のツールを提供するB2B SaaSプラットフォームです。

## あなたの役割
あなたは、問題を調査し、ナレッジベースから解決策を見つけ、明確で実行可能なガイダンスとともにチケットを解決することで、受信したサポートチケットに対応します。

## プロセス
1. 必ず最初にチケットを参照し、状況の全体像を把握すること
2. ナレッジベースを検索し、関連する解決策と手順を探すこと
3. 具体的な次のステップを含む、詳細な解決内容でチケットを解決すること

## ガイドライン
- 徹底すること：問題が単純に見えても、解決する前に必ずKBを検索する
- 具体的にすること：解決内容には、正確な手順、リンク、所要時間を含める
- 必要なときはエスカレーションすること：確信度が低い場合や、特権アクセスが必要な問題の場合は、エスカレーション対象としてマークする
- 正確に分類すること：billing、technical、account、feature_requestのいずれか
- ナレッジベースの記事は英語で書かれているため、search_kbのクエリには英語のキーワードを使うこと

## エスカレーション基準
- $10,000を超える金銭に関する問題
- セキュリティに関わるアカウントの侵害
- エンジニアリングの介入が必要な問題
- Enterprise SLAのお客様（1時間以内に応答）

## TechFlowの製品ティア
- Starter（$29/ユーザー/月）：基本的なプロジェクト管理、5GBのストレージ、メールサポート、最大5プロジェクト、コミュニティフォーラム
- Professional（$79/ユーザー/月）：高度な分析、100GBのストレージ、優先サポート、APIアクセス、プロジェクト数無制限、カスタムフィールド、ガントチャート、時間トラッキング
- Enterprise（個別見積り）：SSO/SAML、ストレージ無制限、専任CSM、カスタム連携、SLA保証、監査ログ、高度なセキュリティ、カスタムブランディング、優先的なAPIレート制限

## 主な問題カテゴリとルーティング
- Billing（請求）：請求書の不一致、支払いの失敗、プランの変更、返金リクエスト、サブスクリプションの解約、日割り計算に関する質問
- Technical（技術）：APIエラー、連携の問題、Webhookの失敗、パフォーマンスの問題、データエクスポートの問題、ブラウザ互換性
- Account（アカウント）：ログインの問題、MFAの問題、SSOの設定、権限の変更、チーム管理、ユーザープロビジョニング
- Feature Requests（機能リクエスト）：製品へのフィードバック、ロードマップの問い合わせ、回避策のリクエスト、ベータ版アクセスのリクエスト

## 回答テンプレート
請求の問題を解決するときは、必ず次を含めること：取引ID、返金の所要期間、確認メールの詳細。
技術的な問題を解決するときは、必ず次を含めること：再現手順、回避策（ある場合）、エスカレーションした場合はエンジニアリングのチケット番号。
アカウントの問題を解決するときは、必ず次を含めること：実施したセキュリティ確認の手順、付与した一時的なアクセス権（ある場合）。

## SLA要件
- Starter：24時間以内に応答、営業時間内のみ
- Professional：4時間以内に応答、延長時間帯（午前6時〜午後10時）
- Enterprise：1時間以内に応答、24/7サポート、専用のSlackチャンネル

## トーン
プロフェッショナルで、共感的で、解決志向であること。解決策に入る前に、お客様の不満を受け止める。お客様名が分かる場合は使う。関連するガイダンスでは、該当する製品ティアに言及する。"""


# TODO: run_agent(user_message)を実装する
# 1. ユーザーメッセージを入れたmessagesリストを作る
# 2. client.messages.create()を次の引数で呼び出す：
#    - model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT, tools=tools
#    - thinking={"type": "adaptive"}
#    - messages=messages
# 3. response.stop_reason == "tool_use"の間、次を繰り返す：
#    a. response.contentをたどり、tool_useブロックを見つける
#    b. execute_tool(block.name, block.input)で各ツールを実行する
#    c. tool_use_idとcontentを持つtool_resultのdictを作る
#    d. アシスタントの応答とツール結果をmessagesに追加する
#       （thinkingブロックも含め、すべてのコンテンツブロックを渡すこと！）
#    e. もう一度APIを呼び出す
# 4. 最終的なresponseを返す

def run_agent(user_message: str):
    """サポートチケットエージェントを実行します。"""
    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model=MODEL,
        max_tokens=32000,
        system=SYSTEM_PROMPT,
        tools=tools,
        thinking={"type": "adaptive"},
        messages=messages
    )

    # ✏️ YOUR TURN（あなたの番）：下の___をすべて埋める（Claudeがまだツールを使いたがっている間、ループを回す）
    while response.stop_reason == ___:  # <-- 「Claudeがツールを呼びたい」ことを表すstop_reasonは？

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": ___,  # <-- この結果をツール呼び出しに結びつける、blockのフィールドはどれ？
                    "content": str(result)
                })

        # TODO: アシスタントのターンとツール結果を、messagesに追加する
        # ポイント：textだけでなく、（thinkingブロックを含む）すべてのコンテンツブロックを渡す
        messages.append({"role": "assistant", "content": ___})  # <-- ここには何が入る？（messagesに追加する値）
        messages.append({"role": "user", "content": tool_results})

        # TODO: 更新したmessagesで、もう一度APIを呼び出す
        response = client.messages.create(
            ___  # <-- 最初のcreate()と同じパラメータ
        )

    return response


# Test it!（試してみましょう）
# response = run_agent("チケット TKT-1042 を解決して")
# for block in response.content:
#     if block.type == "text" and block.text.strip():
#         print(f"\n 最終応答：\n{block.text}")

## パート4：構造化出力を追加する

TechFlowの下流システムには、機械が読める形式の解決結果が必要です。チケット管理プラットフォームはデータベースを更新し、分析ダッシュボードはカテゴリを集計し、エスカレーションのルーターは`escalation_needed`フラグを確認します。自由形式のテキストでは使えません。

`output_config.format`は、Claudeのテキスト応答をJSONスキーマに合わせるよう制約します。**重要：** この形式の制約は*すべての*テキスト出力に適用されるため、ツールループが終わった後の最終API呼び出しにだけ追加します。ループの間、Claudeは形式の制約なしで通常どおりツールを使います。ツールが終わったら、`output_config.format`と`tool_choice={"type": "none"}`を指定してもう1回呼び出し、構造化されたJSONの解決結果を得ます。

**注意：** アダプティブ思考を使うと、レスポンスに`[thinking, text]`のブロックが含まれることがあります。構造化JSONは、*最後*のtextブロックに入っています。

> 📖 **参考：** [構造化出力／JSONモード](https://docs.anthropic.com/en/docs/build-with-claude/structured-outputs)

### ✏️ YOUR TURN（あなたの番）：`run_agent_structured()`を完成させる

ツールループは、すでに書いてあります。最終呼び出しにある2つの`___`（`output_config`と`tool_choice`）を埋めてください。

In [ ]:
# ✅ RESOLUTION_SCHEMAは下で定義済みです。読んでから、run_agent_structured()を実装してください
RESOLUTION_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "diagnosis": {"type": "string", "description": "問題の根本原因の分析"},
            "solution_steps": {"type": "array", "items": {"type": "string"}, "description": "解決のための手順（実行順）"},
            "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
            "escalation_needed": {"type": "boolean"},
            "category": {"type": "string", "enum": ["billing", "technical", "account", "feature_request"]}
        },
        "required": ["diagnosis", "solution_steps", "confidence", "escalation_needed", "category"],
        "additionalProperties": False
    }
}


def get_structured_result(response) -> dict:
    """レスポンスの最後のtextブロックから、構造化JSONを取り出します。"""
    # アダプティブ思考では、contentが[thinking, text]になることがある。JSONは最後のtextブロックにある
    text_blocks = [b for b in response.content if b.type == "text" and b.text.strip()]
    if text_blocks:
        return json.loads(text_blocks[-1].text)
    return None


def run_agent_structured(user_message: str) -> dict:
    """構造化JSON出力でエージェントを実行します。"""
    # Step 1：ツールループを実行する。run_agent()と同じで、ここではoutput_configを渡さない
    # （formatはすべてのテキスト出力を制約するため、有効にするとツールが動かない）
    messages = [{"role": "user", "content": user_message}]
    response = client.messages.create(
        model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
        tools=tools, thinking={"type": "adaptive"}, messages=messages
    )
    while response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
        response = client.messages.create(
            model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
            tools=tools, thinking={"type": "adaptive"}, messages=messages
        )

    # Step 2：構造化出力を得るために、最後にもう1回呼び出す
    messages.append({"role": "user", "content": "構造化された解決結果をJSONで出力してください。"})
    # ✏️ YOUR TURN（あなたの番）：下の2つの___を埋める
    final = client.messages.create(
        model=MODEL,
        max_tokens=8000,
        system=SYSTEM_PROMPT,
        output_config=___,   # <-- ここにRESOLUTION_SCHEMAを渡す。{"format": RESOLUTION_SCHEMA}の形で指定する
        tool_choice=___,     # <-- これ以上のツール呼び出しを防ぐ：{"type": "none"}
        thinking={"type": "adaptive"},
        messages=messages
    )
    return get_structured_result(final)


# result = run_agent_structured("チケット TKT-1042 を解決して")
# print(json.dumps(result, indent=2))

---
### ✅ チェックポイント1：構造化出力を備えた、動くサポートチケットエージェント

チケット1件につき2〜3個のツールを呼び出し、構造化JSONを返すエージェントができているはずです。

**確認：** `run_agent_structured("Resolve ticket TKT-1044")`（日本語版のノートブックでは`run_agent_structured("チケット TKT-1044 を解決して")`）を実行すると、category: feature_requestになり、APIを使った回避策が提案されます。

---
# パート2：アダプティブ思考

基本のエージェントは動きますが、TechFlowのサポートリーダーは懸念を挙げます。「二重請求やパスワードリセットのように、単純なチケットもあります。一方で、本当にあいまいなチケットもあります。難しいチケットでは深く推論してほしいけれど、簡単なチケットの処理が遅くなるのは困ります。それに、難しいチケットで判断を誤ったとき、*なぜ*そう判断したのかが見えません。」

アダプティブ思考は、この両方を解決します。`output_config.effort`パラメータで、Claudeの推論の深さを制御できます。複雑なチケットやあいまいなチケットにはhigh、単純なチケットにはlowを指定します。思考トレースからは、判断のプロセスが見えます。なぜエスカレーションを選んだのか、どのKB記事を最も重視したのかを、後から監査できます。

> 📖 **参考：** [アダプティブ思考](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)

## セル5：エフォートレベルによる思考制御を追加する

`output_config.effort`を追加して、Claudeの推論の深さを制御します。highエフォートでは、ツール呼び出しの合間に詳細な思考トレースが生成されます。何を検索するか、KBの結果をどう評価するか、エスカレーションするかどうかを推論します。lowエフォートでは、単純なチケット向けに推論を短く抑えます。思考ブロックを表示して、エージェントの判断プロセスを確認しましょう。

### ✏️ YOUR TURN（あなたの番）：`run_agent_thinking()`を実装する

今回は関数全体を自分で書きます。コメントが、4つのステップを案内します。

In [ ]:
# TODO: エージェントに、エフォートレベルによる思考制御を追加する
# 1. run_agentをコピーし、thinking={"type": "adaptive"}でツールループを実行する
#    と output_config={"effort": effort} を渡す（formatは渡さない。最後の呼び出しまで取っておく）
# 2. ループの中で、思考ブロックを表示する：block.type == "thinking"
# 3. tool loopが終わったら、次の指定で最終呼び出しを行う：
#    - output_config={"effort": effort, "format": RESOLUTION_SCHEMA}
#    - tool_choice={"type": "none"}
#    - 「構造化された解決結果をJSONで出力してください。」のようなユーザーメッセージを追加する
# 4. get_structured_result()を使って、最終レスポンスを解析する

def run_agent_thinking(user_message: str, effort: str = "high") -> dict:
    """エフォートレベルで制御されるアダプティブ思考付きで、エージェントを実行します。"""
    # ✏️ YOUR TURN（あなたの番）：ここに実装を書く
    pass

## セル6：アダプティブ思考の動きを確認する

MedTech Solutions（TKT-1046）から、断続的な500エラーの報告が来ています。APIコールの15%が失敗し、先方では何も変更しておらず、午前2時に始まりました。レート制限でしょうか。サーバー側の障害でしょうか。ネットワークの問題でしょうか。KBに完全に一致する記事はありません。

ここが、思考の真価が出る場面です。あいまいなチケットをhighエフォートで実行し、次に同じチケットをlowエフォートで実行して比べてください。推論の深さがどう変わるかを観察します。そのうえで考えてみてください。TechFlowの1日500件のチケットで、単純なチケットと複雑なチケットを、異なるエフォートレベルにどう振り分けますか？

In [ ]:
def _not_built_yet(fn_name, result):
    """ビルドアロング用のガード：YOUR TURNのスタブは、実装するまでNoneを返します。"""
    if result is not None:
        return False
    print(f"\n[--] {fn_name}() isn't built yet - that's the exercise, not a bug.")
    print(f"     Find the '# YOUR TURN' marker inside {fn_name}(), build it, then run this again.")
    return True

# あいまいなチケットをhighエフォートで実行し、思考トレースを観察する
print("=== TKT-1046：断続的なAPIエラー（判断が難しいチケット） ===\n")
result = run_agent_thinking("チケット TKT-1046 を解決して", effort="high")
if _not_built_yet("run_agent_thinking", result):
    raise SystemExit(0)
print(f"\nResolution:")
print(json.dumps(result, indent=2))

# 次に比較：同じチケットをlowエフォートで実行する
print(f"\n\n{'='*50}")
print("=== 同じチケット、LOWエフォート ===")
print(f"{'='*50}\n")

for effort in ["high", "low"]:
    start = time.time()
    result = run_agent_thinking("チケット TKT-1046 を解決して", effort=effort)
    elapsed = time.time() - start
    print(f"\n[effort={effort}] Confidence: {result['confidence']} | Steps: {len(result['solution_steps'])} | Escalate: {result['escalation_needed']} | Time: {elapsed:.1f}s")

---
### ✅ チェックポイント2：推論の可視化とエフォート制御を備えたエージェント

エフォートを比べると、推論の深さと質に目に見える違いが出るはずです。

---
# パート3：ストリーミング

TechFlowのサポート担当者は、リアルタイムのダッシュボードでAIによる振り分けを監視しています。チケットが届いたとき、エージェントが作業している様子を*見る*必要があります。どのチケットを参照しているのか、何を検索しているのか、どう推論しているのか。15秒のスピナーの後に長い文章が一気に出ても、信頼は得られません。

それを解決するのがストリーミングです。`create()`を`stream()`に置き換えると、トークンがリアルタイムに届きます。エージェントが推論している間は思考トレースが流れ、ツール呼び出しは実行されるたびに表示され、構造化された解決結果は最後にストリームされます。

> 📖 **参考：** [ストリーミングメッセージ](https://docs.anthropic.com/en/docs/build-with-claude/streaming)

## セル7：ストリーミングのエージェントループ

`client.messages.create()`を`client.messages.stream()`に置き換えて、TechFlowのダッシュボードにエージェントの作業をリアルタイムで表示できるようにします。主なイベントタイプを処理します。`thinking_delta`（推論のトークン）、`text_delta`（最終応答）、`input_json_delta`（ツールの引数）です。

ストリームの完了後は、`stream.get_final_message()`を使って、ループを続けるための完全なレスポンスオブジェクトを取得します。

### ✏️ YOUR TURN（あなたの番）：`run_agent_streaming()`を実装する

関数全体を自分で書きます。`create()`を`stream()`に置き換え、コメントに挙げたストリームイベントを処理してください。

In [ ]:
# TODO: ストリーミングのエージェントループを作る
# 1. create()をstream()に置き換え、コンテキストマネージャ（with ... as stream:）を使う
#    ツールループの間は output_config={"effort": effort} を使う（formatの制約はなし）
# 2. stream eventを順に処理し、次を扱う：
#    - content_block_start：content_block.typeを確認する（thinking/tool_use/text）
#    - content_block_delta：thinking_delta、text_delta、input_json_deltaを処理する
# 3. ストリーミングの後、stream.get_final_message()で完全なレスポンスを取得する
# 4. stop_reasonがtool_useなら、ツールを実行してループを続ける
# 5. tool loopが終わったら、次の指定で、最終のstreaming呼び出しを行う：
#    - output_config={"effort": effort, "format": RESOLUTION_SCHEMA}
#    - tool_choice={"type": "none"}
# 6. 最終的なJSONには、get_structured_result()を使う
# 忘れずに：stream()に thinking={"type": "adaptive", "display": "summarized"} を渡す

def run_agent_streaming(user_message: str, effort: str = "high") -> dict:
    """ストリーミング出力付きで、エージェントを実行します。"""
    # ✏️ YOUR TURN（あなたの番）：ここに実装を書く
    pass

## セル8：フルデモ

SecureNet Systemsから、重大なチケットが届きました。管理者アカウントがロックアウトされ、47人のチームメンバーが作業できません。ストリーミングを使ってエージェントをフルで実行し、4つの機能がリアルタイムに組み合わさる様子を確認します。エージェントループがツール呼び出しを取りまとめ、構造化出力が解決結果の形式を保証し、アダプティブ思考がセキュリティ上の影響を推論し、ストリーミングがそのすべてを進行中に見せてくれます。

In [ ]:
print("エージェントのフルデモ：TKT-1045を解決する（アカウントのロックアウト）")
print("   ストリーミング + アダプティブ思考 + ツール + 構造化出力")
print("=" * 60)

start = time.time()
result = run_agent_streaming("チケット TKT-1045 を解決して")
elapsed = time.time() - start
if _not_built_yet("run_agent_streaming", result):
    raise SystemExit(0)

print(f"\n\n{'=' * 60}")
print(f"Total time: {elapsed:.1f}s")
print(f"\nStructured Resolution:")
print(json.dumps(result, indent=2))

---
### ✅ チェックポイント3：リアルタイムのエージェント

4つの機能が、1回のエージェント実行の中で組み合わさっています。
エージェントループ + 構造化出力 + アダプティブ思考 + ストリーミング。

---
# 発展課題

1. **ツール選択の制御**：`tool_choice: {"type": "none"}`を使って、Claudeにツール呼び出しをやめさせる
2. **エフォートの最適化**：すべてのチケットを`effort="high"`、`effort="medium"`、`effort="low"`で処理し、品質と速度の表を作る
3. **バッチ処理**：Batch APIを使って、すべてのチケットを一度に処理する（コスト50%削減）

## さらに学ぶ

- [Claude APIドキュメント](https://docs.anthropic.com/en/docs)
- [ツール使用ガイド](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [アダプティブ思考](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)
- [構造化出力](https://docs.anthropic.com/en/docs/build-with-claude/structured-outputs)
- [ストリーミング](https://docs.anthropic.com/en/docs/build-with-claude/streaming)